# 02.5 — QA random sampling & CoDE (expert collaboration) architecture

This notebook does two things:

1) **Sample many random QA pairs** (content, quality score, which experts were used).
2) **Describe the CoDE pipeline** (flow, modules, expert list, data field conventions).

Assumes project root `VeritasCarbon_VLDB/`; if opened from `notebooks/` it will still resolve the root.

## CoDE architecture (concepts and code)

### 1) Goal
CoDE (Council of Domain Experts / Council of Domain Experts) is used in this project to:
- **Adaptively select expert perspectives per chunk** (e.g. extraction / classification / analysis / standard alignment).
- **Inject domain knowledge** (from `data/knowledge_base/`).
- Generate **high-quality QA/instruction data** that is more controllable and carbon/ESG-oriented under expert constraints.

### 2) CoDE pipeline (steps)
Each run can be seen as a traceable pipeline; every step produces persistable intermediate information (especially `metadata`):
1. **Input**: Read `chunk_text` (and `chunk_id` / `doc_id`).
2. **Feature extraction** (selector pre-step): Extract length, numbers/entities, temporal words, standard/compliance words, comparison words, etc., from the chunk.
3. **Layered expert selection (ExpertSelector)**: One base-layer expert is required; analysis/verification/graph layers are added by threshold.
4. **Domain knowledge retrieval (DomainKnowledgeInjector)**: Retrieve Top-K knowledge snippets from `data/knowledge_base/` by chunk keywords/entities/topics.
5. **Expert collaboration (COEFramework)**: Assemble "chunk + selected expert roles + retrieved knowledge" into structured context and run sequential/parallel collaboration.
6. **Meta-Expert (optional)**: Combine multiple expert opinions to produce final constraints and prompts (1–2 iterations).
7. **LLM Q/A generation**: Generate questions and answers in the expert+knowledge+constraint context and write to `metadata`.
8. **Quality evaluation/scoring**: Output `quality_score` (and optionally record failure reasons).
9. **Persist**: Write JSONL + checkpoint; supports resume and later analysis.

#### 2.1 How `quality_score` is computed (current implementation, from code)
`quality_score` is computed by `_evaluate_quality(output, chunk_text)` in `src/instruction_generation/coe_framework_02_03.py`, clamped to $[0,1]$. It is a **heuristic score** (not the multi-metric evaluator in `evaluation_metrics_02_06.py`). Rules:
- **Length gate (direct reject)**: If `instruction` length ≤ 10 or `response` length ≤ 20, return `0.0`.
- **Length base score**: After passing the length gate, add `0.3`.
- **Relevance score (cap 0.4)**: Use first 200 chars of `chunk_text`; compute 3-gram set overlap with response:
  - Let $C$ = chunk 3-gram set, $R$ = response 3-gram set; $overlap = |C \cap R| / |C|$.
  - Add `min(overlap * 0.4, 0.4)`.
  - If chunk is too short for 3-grams, fallback: count chars from first 50 chars of chunk that appear in response ($common\_chars$), add `min(common_chars/50 * 0.4, 0.4)`.
- **Completeness score (cap 0.3)**: If `len(response) > 50` add `0.2`; if `len(response) > 100` add another `0.1`.
- **Cap**: Return `min(total_score, 1.0)`.

Note: `COEFramework` uses `quality_threshold` (default `0.7`) to decide whether to continue iterating; when `quality_score >= quality_threshold` it stops early.

### 3) Mapping to code files
- Expert selection: `src/instruction_generation/expert_selector_02_01.py`
- Domain knowledge injection: `src/instruction_generation/domain_knowledge_02_02.py`
- CoDE orchestration: `src/instruction_generation/coe_framework_02_03.py`
- Expert set and implementations: `src/instruction_generation/expert_agents_02_04.py`
- Instruction/QA generation: `src/instruction_generation/instruction_generator_02_05.py`
- Evaluation metrics: `src/instruction_generation/evaluation_metrics_02_06.py`
- Baselines and ablation: `src/instruction_generation/baseline_experiments_02_07.py`, `ablation_studies_02_08.py`
- Meta-Expert: `src/instruction_generation/meta_expert_02_09.py`

### 4) Expert list (by layer)
The "definitions" below come from `EXPERT_TYPES` in `src/instruction_generation/expert_selector_02_01.py` (the system’s official role for each expert).

**Base layer (5)**
- `qa_expert` (QA): Good at generating Q&A pairs. Typical signals: question words/patterns, factual content (`question_keywords`, `factual_content`).
- `summary_expert` (Summary): Good at summarization. Typical signals: long text, structured content (`long_text`, `structured_content`).
- `extraction_expert` (Extraction): Good at extracting metrics, entities, numbers. Typical signals: `structured_data`, `numbers`, `entities`.
- `classification_expert` (Classification): Good at classification (themes, ESG dimensions). Typical signals: categorical content, high ESG keyword density (`categorical_content`, `esg_keywords`).
- `analysis_expert` (Analysis): Good at comparative analysis. Typical signals: comparative semantics, multiple entities (`comparative_content`, `multiple_entities`).

**Analysis layer (3)**
- `temporal_analysis_expert`: Cross-year comparison and trend analysis. Signals: temporal/year terms, trend words (`temporal_keywords`, `year_mentions`, `trend_indicators`).
- `benchmark_comparison_expert`: Benchmark/peer comparison. Signals: industry terms, comparison words, benchmark phrasing (`industry_keywords`, `comparative_keywords`, `benchmark_indicators`).
- `greenwashing_detection_expert`: Detecting greenwashing (claims vs verifiable info). Signals: vague commitments, promise wording, verification cues (`vague_commitments`, `promise_keywords`, `verification_indicators`).

**Verification layer (2)**
- `consistency_verification_expert`: Data consistency (e.g. consistent definitions, no contradictory numbers). Signals: numbers, data-related words, comparison words.
- `standard_alignment_expert`: ESG standards and compliance alignment. Signals: standard keywords, compliance cues, structured phrasing.

**Graph layer (1)**
- `knowledge_graph_expert`: Entity and relation extraction for graph structures. Signals: entities, relationship cues, structured data.

> Note: The "definition/signals" above describe each expert’s **role**. Whether an expert is actually selected depends on the layered selection rules and thresholds in section 5 (and `max_experts` truncation).

### 5) Current "trigger signals" (reflects actual implementation)
This section is derived from **current code** in `src/instruction_generation/expert_selector_02_01.py` (rules and layered strategy), not an idealized design. Treat it as the authoritative description of how experts are selected.

#### 5.1 Selection method in use
- With `use_layered_selection=True`, `ExpertSelector.select_experts()` uses **`select_experts_layered()`**.
- The ML classifier is currently a placeholder (falls back to rules) and does not change the path when layered selection is on.
- In `configs/config.yaml`: `coe.use_layered_selection: true`, with:
  - `analysis_layer_threshold = 0.3`
  - `verification_layer_threshold = 0.3`
  - `graph_layer_threshold = 0.5`
  - `max_experts = 3` (critical: even when multiple layers trigger, the list is truncated).

#### 5.2 How trigger features are computed
All features are normalized to $[0,1]$; higher = stronger signal. (Keyword lists in the code include Chinese and English terms; the logic is count-based and normalized.)
- `has_question_keywords`: count of question-style keywords (e.g. what, how, why, which, how many, whether) — normalized, cap 1.
- `esg_keyword_density`: count of ESG keywords (environment, society, governance, ESG, CSR, sustainability, carbon, etc.) — normalized.
- `has_structured_data`: count of structure indicators (colons, commas, parentheses, numbered items, etc.) — count/10, cap 1.
- `has_numbers`: count of digit spans — count/10, cap 1.
- `has_entities`: count of entity keywords (company, enterprise, carbon, employees, etc.) — count/5, cap 1.
- `comparative_keywords`: count of comparison words (compare, difference, better than, etc.) — count/3, cap 1.
- `temporal_keywords`: count of time/trend words (year, quarter, trend, growth, etc.) — count/5, cap 1.
- `industry_keywords`: count of industry/benchmark words — count/4, cap 1.
- `vague_commitments`: count of vague-commitment words (commit to, plan to, target, etc.) — count/4, cap 1.
- `standard_keywords`: count of standard/compliance words (GRI, TCFD, SASB, ISO, CDP, standard, framework, compliance, etc.) — count/5, cap 1.

#### 5.3 Layered selection: when each expert is added (current logic)
**A) Base layer (exactly 1)**  
Scores for the five base experts are computed; the highest is chosen. No hard threshold — **relative competition**.
- `qa_expert`: `0.5 * has_question_keywords + 0.5 * esg_keyword_density`
- `summary_expert`: `0.5 * length_signal + 0.5 * has_structured_data` (length_signal = 1.0 if length>300 else length/300)
- `extraction_expert`: `0.4 * has_structured_data + 0.3 * has_numbers + 0.3 * has_entities`
- `classification_expert`: `0.7 * esg_keyword_density + 0.3 * has_entities`
- `analysis_expert`: `0.6 * comparative_keywords + 0.4 * has_entities`

**B) Analysis layer (up to 2 added)**  
If above threshold, add in fixed order; then truncate to at most 2.
- If `temporal_keywords > 0.3` ⇒ add `temporal_analysis_expert`
- If `industry_keywords > 0.3` ⇒ add `benchmark_comparison_expert`
- If `vague_commitments > 0.3` ⇒ add `greenwashing_detection_expert`  
Trigger in this layer uses **one feature per expert** (e.g. benchmark expert only uses `industry_keywords`).

**C) Verification layer (at most 1, with priority)**  
- If `has_numbers > 0.3` ⇒ add `consistency_verification_expert`
- Else if `standard_keywords > 0.3` ⇒ add `standard_alignment_expert`  
So numeric signal wins over standard keywords.

**D) Graph layer (at most 1)**  
- If `has_entities > 0.5` **and** `has_structured_data > 0.3` ⇒ add `knowledge_graph_expert`  
Still subject to `max_experts` truncation.

**E) Total truncation**  
`selected = selected[:max_experts]` with `max_experts = 3`. So:
- Experts that trigger in later layers may be dropped by the cap.
- If an expert does not appear in a sample, it may be "triggered but truncated" rather than "not triggered".

#### 5.4 Expert present in code but never selected
`promise_performance_verification_expert` exists in `EXPERT_TYPES` but is not wired into the rule/layered selection (commented out or not in the flow), so it does not appear in `selected_experts`.

### 6) QA JSONL field conventions (this notebook supports multiple versions)
Common fields:
- `question` / `answer`
- `chunk_id`, `doc_id`
- `metadata.quality_score` (or top-level `quality_score`)
- `metadata.selected_experts` (or `metadata.experts` / `selected_experts`)

If your JSONL uses slightly different names, this notebook’s `extract_fields()` handles them.

In [1]:
from __future__ import annotations

import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple


def get_project_root() -> Path:
    """Best-effort locate repo root by presence of configs/ and src/."""
    p = Path.cwd()
    if p.name == 'notebooks':
        p = p.parent
    cur = p
    while cur != cur.parent:
        if (cur / 'src').exists() and (cur / 'configs').exists():
            return cur
        cur = cur.parent
    return p


PROJECT_ROOT = get_project_root()
PROJECT_ROOT

PosixPath('/hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB')

In [2]:
# Pick a QA JSONL file (auto-detect by priority; or set manually)

CANDIDATES = [
    PROJECT_ROOT / 'data' / 'instructions' / 'qa_pairs_complete_v3.jsonl',
    PROJECT_ROOT / 'data' / 'instruction_datasets' / 'qa_pairs_qwen72b.jsonl',
    PROJECT_ROOT / 'data' / 'instruction_datasets' / 'qa_pairs_complete_v3.jsonl',
]

QA_FILE = next((p for p in CANDIDATES if p.exists()), None)
QA_FILE

PosixPath('/hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB/data/instructions/qa_pairs_complete_v3.jsonl')

In [3]:
# If auto-detect fails, set path manually here
# QA_FILE = PROJECT_ROOT / 'data' / 'instructions' / 'qa_pairs_complete_v3.jsonl'

if QA_FILE is None:
    raise FileNotFoundError(
        'No QA JSONL found. Set QA_FILE manually and ensure file exists under data/.'
    )

print('Using QA file:', QA_FILE)
print('Size (MB):', QA_FILE.stat().st_size / 1024 / 1024)

Using QA file: /hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB/data/instructions/qa_pairs_complete_v3.jsonl
Size (MB): 88.14036655426025


## Random sample: QA content, quality score, expert selection

This cell uses **reservoir sampling** to stream and randomly sample `K` records from a large file:
- No need to load the whole JSONL into memory
- More stable for very large files

You can adjust:
- `K`: number of samples
- `SEED`: random seed (for reproducibility)
- `MAX_CHARS`: max text length to display (avoid notebook lag)
- `MIN_QUALITY`: only show samples with quality ≥ threshold (optional)

In [ ]:
@dataclass
class QARecord:
    question: str
    answer: str
    chunk_text: str
    quality_score: Optional[float]
    selected_experts: List[str]
    expert_names: List[str]
    chunk_id: str
    doc_id: str
    raw: Dict[str, Any]


def _safe_float(x: Any) -> Optional[float]:
    try:
        if x is None:
            return None
        return float(x)
    except Exception:
        return None


def extract_fields(obj: Dict[str, Any]) -> QARecord:
    # Question / Answer
    question = obj.get('question') or obj.get('instruction') or ''
    answer = obj.get('answer') or obj.get('output') or ''

    # IDs
    chunk_id = obj.get('chunk_id', '')
    doc_id = obj.get('doc_id', '')

    # metadata container
    md = obj.get('metadata') if isinstance(obj.get('metadata'), dict) else {}

    # Chunk text (best-effort; different generators use different keys)
    chunk_text = (
        obj.get('chunk')
        or obj.get('text')
        or md.get('chunk')
        or md.get('chunk_text')
        or md.get('text')
        or ''
    )

    # Quality score
    quality = _safe_float(md.get('quality_score'))
    if quality is None:
        quality = _safe_float(obj.get('quality_score'))

    # Experts
    selected_experts: List[str] = []
    expert_names: List[str] = []

    for key in ('selected_experts', 'experts', 'expert_ids'):
        v = md.get(key)
        if isinstance(v, list):
            selected_experts = [str(x) for x in v]
            break
    if not selected_experts:
        v2 = obj.get('selected_experts')
        if isinstance(v2, list):
            selected_experts = [str(x) for x in v2]

    v_names = md.get('expert_names')
    if isinstance(v_names, list):
        expert_names = [str(x) for x in v_names]

    # fallback ids nested in metadata
    if not chunk_id:
        chunk_id = str(md.get('chunk_id', ''))
    if not doc_id:
        doc_id = str(md.get('doc_id', ''))

    return QARecord(
        question=str(question),
        answer=str(answer),
        chunk_text=str(chunk_text),
        quality_score=quality,
        selected_experts=selected_experts,
        expert_names=expert_names,
        chunk_id=str(chunk_id),
        doc_id=str(doc_id),
        raw=obj,
    )


def iter_jsonl(path: Path) -> Iterable[Dict[str, Any]]:
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)


def reservoir_sample_jsonl(
    path: Path,
    k: int,
    seed: int = 42,
    min_quality: Optional[float] = None,
    max_scan: Optional[int] = None,
) -> List[QARecord]:
    rnd = random.Random(seed)
    reservoir: List[QARecord] = []
    seen = 0

    for i, obj in enumerate(iter_jsonl(path), start=1):
        if max_scan is not None and i > max_scan:
            break
        rec = extract_fields(obj)
        if min_quality is not None:
            if rec.quality_score is None or rec.quality_score < min_quality:
                continue

        seen += 1
        if len(reservoir) < k:
            reservoir.append(rec)
        else:
            j = rnd.randint(1, seen)
            if j <= k:
                reservoir[j - 1] = rec

    rnd.shuffle(reservoir)
    return reservoir


def _record_key(rec: QARecord) -> str:
    # Prefer stable IDs if present
    if rec.chunk_id:
        return f"chunk_id::{rec.chunk_id}"
    # Fallback (may be longer but OK for small samples)
    return f"qa::{rec.question}||{rec.answer}"


def sample_covering_experts_jsonl(
    path: Path,
    k: int,
    target_experts: List[str],
    seed: int = 42,
    min_quality: Optional[float] = None,
    max_scan: Optional[int] = None,
) -> Tuple[List[QARecord], List[str]]:
    """Sample k records while ensuring coverage over target_experts (if present in data).

    Strategy:
      - One streaming pass.
      - Maintain a 'coverage' dict: expert -> first-seen record that uses this expert.
      - Maintain a global reservoir of size k for the rest.
      - Final samples = (one per expert in target_experts, de-duplicated) + fill from reservoir.

    Returns:
      samples, missing_experts
    """
    if k < len(target_experts):
        raise ValueError(
            f"K={k} is less than the number of expert types={len(target_experts)}; cannot cover all experts. Set K >= {len(target_experts)}"
        )

    rnd = random.Random(seed)
    target_set = set(target_experts)

    coverage: Dict[str, QARecord] = {}
    reservoir: List[QARecord] = []
    seen = 0

    for i, obj in enumerate(iter_jsonl(path), start=1):
        if max_scan is not None and i > max_scan:
            break
        rec = extract_fields(obj)

        if min_quality is not None:
            if rec.quality_score is None or rec.quality_score < min_quality:
                continue

        # coverage update
        for e in rec.selected_experts:
            if e in target_set and e not in coverage:
                coverage[e] = rec

        # global reservoir
        seen += 1
        if len(reservoir) < k:
            reservoir.append(rec)
        else:
            j = rnd.randint(1, seen)
            if j <= k:
                reservoir[j - 1] = rec

    # Build final sample set
    selected: List[QARecord] = []
    used_keys: set[str] = set()

    def _add(rec: QARecord) -> None:
        key = _record_key(rec)
        if key in used_keys:
            return
        used_keys.add(key)
        selected.append(rec)

    missing: List[str] = []
    for e in target_experts:
        if e in coverage:
            _add(coverage[e])
        else:
            missing.append(e)

    rnd.shuffle(reservoir)
    for rec in reservoir:
        if len(selected) >= k:
            break
        _add(rec)

    rnd.shuffle(selected)
    return selected, missing


def expert_coverage_counts(samples: List[QARecord], target_experts: List[str]) -> Dict[str, int]:
    counts: Dict[str, int] = {e: 0 for e in target_experts}
    for rec in samples:
        for e in rec.selected_experts:
            if e in counts:
                counts[e] += 1
    return counts


def shorten(text: str, max_chars: int) -> str:
    if text is None:
        return ''
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3] + '...'


In [5]:
# ====== Sampling parameters (edit as needed) ======
# Goal: sample so that all expert types are covered (each at least once, if present in data)
# Plus: exactly reproduce selected_experts at generation time — same ExpertSelector code and layered_config.

# Target expert list (default 11; adjust if your project adds/removes experts)
TARGET_EXPERTS = [
    # Base layer
    "qa_expert",
    "summary_expert",
    "extraction_expert",
    "classification_expert",
    "analysis_expert",
    # Analysis layer
    "temporal_analysis_expert",
    "benchmark_comparison_expert",
    "greenwashing_detection_expert",
    # Verification layer
    "consistency_verification_expert",
    "standard_alignment_expert",
    # Graph layer
    "knowledge_graph_expert",
]

K = 30
SEED = 20
MAX_CHARS = 1200
MIN_QUALITY = None   # e.g. 0.7 for high quality only
MAX_SCAN = None      # e.g. 200000 to speed up (scan first 200k lines)

# Output CSV (chunk + Q + A + quality + experts + repro check)
OUTPUT_CSV = PROJECT_ROOT / "results" / "outputs" / f"qa_samples_02_5_seed{SEED}_k{K}.csv"

# ====== Exact reproduction: build ExpertSelector matching generation time ======
# Note: In next_30k notebook, ExpertSelector layered_config is hard-coded (0.6/0.7/0.8);
# configs/config.yaml uses 0.3/0.3/0.5. Mismatch causes inconsistent re-runs.
# Options:
#   - REPRO_PROFILE='auto': compare both configs on a small sample, pick the one that matches generation time.
#   - REPRO_PROFILE='next_30k': force 0.6/0.7/0.8 (reproduce next_30k generation logic).
#   - REPRO_PROFILE='config_yaml': force config.yaml (reproduce config-based generation).

REPRO_PROFILE = "auto"  # 'auto' | 'next_30k' | 'config_yaml'
AUTO_EVAL_N = 1200        # In auto mode, number of rows to scan for profile selection (avoid slow full scan)

import sys
import yaml
import json

# Ensure import path matches generation time (next_30k notebook uses instruction_generation.*)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))

_cfg: dict = {}
_cfg_path = PROJECT_ROOT / "configs" / "config.yaml"
if _cfg_path.exists():
    with open(_cfg_path, "r", encoding="utf-8") as f:
        _cfg = yaml.safe_load(f) or {}

_coe = (_cfg.get("coe") or {}) if isinstance(_cfg, dict) else {}
_layered = (_coe.get("layered_selection") or {}) if isinstance(_coe, dict) else {}

CFG_ANALYSIS_TH = float(_layered.get("analysis_layer_threshold", 0.3))
CFG_VERIF_TH = float(_layered.get("verification_layer_threshold", 0.3))
CFG_GRAPH_TH = float(_layered.get("graph_layer_threshold", 0.5))
CFG_MAX_EXPERTS = int(_coe.get("max_experts", 3))
CFG_MIN_EXPERTS = int(_coe.get("min_experts", 1))
CFG_BASE_REQUIRED = bool(_layered.get("base_layer_required", True))

# Hard-coded thresholds from next_30k notebook (see 02_InstructionGeneration_v3_next_30k.ipynb ExpertSelector init cell)
NEXT30K_ANALYSIS_TH = 0.6
NEXT30K_VERIF_TH = 0.7
NEXT30K_GRAPH_TH = 0.8
NEXT30K_MAX_EXPERTS = 3
NEXT30K_MIN_EXPERTS = 1
NEXT30K_BASE_REQUIRED = True

def _build_selector(profile: str):
    try:
        from instruction_generation.expert_selector_02_01 import ExpertSelector  # type: ignore
    except Exception as e:
        print("Cannot import instruction_generation.expert_selector_02_01.ExpertSelector")
        print("   Error:", e)
        raise

    if profile == "config_yaml":
        layered = {
            "base_layer_required": CFG_BASE_REQUIRED,
            "analysis_layer_threshold": CFG_ANALYSIS_TH,
            "verification_layer_threshold": CFG_VERIF_TH,
            "graph_layer_threshold": CFG_GRAPH_TH,
        }
        return ExpertSelector(
            use_ml_classifier=False,
            min_experts=CFG_MIN_EXPERTS,
            max_experts=CFG_MAX_EXPERTS,
            use_layered_selection=True,
            layered_config=layered,
        ), {"profile": profile, **layered, "min_experts": CFG_MIN_EXPERTS, "max_experts": CFG_MAX_EXPERTS}

    if profile == "next_30k":
        layered = {
            "base_layer_required": NEXT30K_BASE_REQUIRED,
            "analysis_layer_threshold": NEXT30K_ANALYSIS_TH,
            "verification_layer_threshold": NEXT30K_VERIF_TH,
            "graph_layer_threshold": NEXT30K_GRAPH_TH,
        }
        return ExpertSelector(
            use_ml_classifier=False,
            min_experts=NEXT30K_MIN_EXPERTS,
            max_experts=NEXT30K_MAX_EXPERTS,
            use_layered_selection=True,
            layered_config=layered,
        ), {"profile": profile, **layered, "min_experts": NEXT30K_MIN_EXPERTS, "max_experts": NEXT30K_MAX_EXPERTS}

    raise ValueError(f"Unknown profile: {profile}")

def _exact_list(v):
    if v is None:
        return []
    if isinstance(v, list):
        return [str(x) for x in v]
    # Some files may store as 'a|b|c'; handle gracefully
    if isinstance(v, str):
        return [p for p in v.split("|") if p]
    return []

def _auto_pick_profile(path: Path, n: int = 1200) -> str:
    # Compare two profiles on the first n rows using exact-match (list equality, order-sensitive)
    cand_profiles = ["next_30k", "config_yaml"]
    scores = {}
    details = {}
    for profile in cand_profiles:
        selector_tmp, _ = _build_selector(profile)
        total = 0
        hit = 0
        for i, obj in enumerate(iter_jsonl(path), start=1):
            if i > n:
                break
            rec = extract_fields(obj)
            sel = _exact_list(rec.selected_experts)
            if not sel or not rec.chunk_text:
                continue
            pred, _ = selector_tmp.select_experts(rec.chunk_text)
            total += 1
            if _exact_list(pred) == sel:
                hit += 1
        scores[profile] = (hit / total) if total else 0.0
        details[profile] = (hit, total)
    best = max(scores.items(), key=lambda x: x[1])[0]
    print("[auto] profile match rate (first N rows, exact list match):")
    for p in cand_profiles:
        h, t = details[p]
        print(f"  - {p:<10} {h}/{t} = {scores[p]*100:.2f}%")
    print(f"[auto] selected profile = {best}")
    return best

# Select reproduction profile
if REPRO_PROFILE == "auto":
    CHOSEN_PROFILE = _auto_pick_profile(QA_FILE, n=AUTO_EVAL_N)
else:
    CHOSEN_PROFILE = REPRO_PROFILE

selector, selector_info = _build_selector(CHOSEN_PROFILE)
print("✓ Loaded ExpertSelector for exact reproduction")
print("  selector_info:", selector_info)

# ====== Sampling (covering all experts) ======
samples, missing_experts = sample_covering_experts_jsonl(
    QA_FILE,
    k=K,
    target_experts=TARGET_EXPERTS,
    seed=SEED,
    min_quality=MIN_QUALITY,
    max_scan=MAX_SCAN,
)

# Coverage summary (from selected_experts in file)
cov = expert_coverage_counts(samples, TARGET_EXPERTS)
covered = [e for e, c in cov.items() if c > 0]

print(f"Sampling done: {len(samples)} records")
print(f"Expert coverage (from file selected_experts): {len(covered)}/{len(TARGET_EXPERTS)}")
if missing_experts:
    print("Experts not found in scan range (missing in data or limited by MAX_SCAN):")
    for e in missing_experts:
        print("  -", e)

print("\nExpert coverage count:")
for e in TARGET_EXPERTS:
    print(f"  {e:<32} {cov[e]}")

# ====== Export CSV (with reproduction columns) ======
import csv

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

n_recompute = 0
n_exact_list_match = 0

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "idx",
            "quality_score",
            "selected_experts",
            "expert_names",
            "chunk_id",
            "doc_id",
            "chunk_text",
            "question",
            "answer",
            # reproduction columns
            "repro_profile",
            "repro_layered_config",
            "recomputed_selected_experts",
            "recomputed_method",
            "selected_minus_recomputed",
            "recomputed_minus_selected",
            "exact_list_match",
        ],
    )
    writer.writeheader()

    for idx, rec in enumerate(samples, start=1):
        sel_list = _exact_list(rec.selected_experts)
        pred_list: list[str] = []
        pred_method = ""
        if rec.chunk_text:
            pred_list, pred_reasons = selector.select_experts(rec.chunk_text)
            pred_list = _exact_list(pred_list)
            pred_method = str((pred_reasons or {}).get("method", ""))
            n_recompute += 1
            if pred_list == sel_list:
                n_exact_list_match += 1

        sel_set = set(sel_list)
        pred_set = set(pred_list)
        selected_minus = sorted(sel_set - pred_set)
        recomputed_minus = sorted(pred_set - sel_set)
        exact_list_match = bool(pred_list == sel_list) if rec.chunk_text else False

        writer.writerow(
            {
                "idx": idx,
                "quality_score": rec.quality_score,
                "selected_experts": "|".join(sel_list),
                "expert_names": "|".join(rec.expert_names),
                "chunk_id": rec.chunk_id,
                "doc_id": rec.doc_id,
                "chunk_text": rec.chunk_text,
                "question": rec.question,
                "answer": rec.answer,
                "repro_profile": CHOSEN_PROFILE,
                "repro_layered_config": json.dumps(selector_info, ensure_ascii=False),
                "recomputed_selected_experts": "|".join(pred_list),
                "recomputed_method": pred_method,
                "selected_minus_recomputed": "|".join(selected_minus),
                "recomputed_minus_selected": "|".join(recomputed_minus),
                "exact_list_match": int(exact_list_match),
            }
        )

print("\nCSV exported:", OUTPUT_CSV)
if n_recompute > 0:
    print(f"Reproduced samples: {n_recompute}")
    print(f"exact list match (order-consistent): {n_exact_list_match}/{n_recompute} ({n_exact_list_match/n_recompute*100:.2f}%)")
else:
    print("No reproduction performed (chunk_text may be empty)")

len(samples)

[auto] profile match rate (first N rows, exact list match):
  - next_30k   1200/1200 = 100.00%
  - config_yaml 367/1200 = 30.58%
[auto] selected profile = next_30k
✓ Loaded ExpertSelector for exact reproduction
  selector_info: {'profile': 'next_30k', 'base_layer_required': True, 'analysis_layer_threshold': 0.6, 'verification_layer_threshold': 0.7, 'graph_layer_threshold': 0.8, 'min_experts': 1, 'max_experts': 3}
Sampling done: 30 records
Expert coverage (from file selected_experts): 11/11

Expert coverage count:
  qa_expert                        2
  summary_expert                   22
  extraction_expert                1
  classification_expert            3
  analysis_expert                  2
  temporal_analysis_expert         1
  benchmark_comparison_expert      1
  greenwashing_detection_expert    4
  consistency_verification_expert  17
  standard_alignment_expert        1
  knowledge_graph_expert           1

CSV exported: /hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB/result

30

In [6]:
# Print sampled records (chunk, Q/A, quality score, experts)

for idx, rec in enumerate(samples, start=1):
    print('=' * 100)
    print(f'[{idx}/{len(samples)}] quality={rec.quality_score}  experts={rec.selected_experts or rec.expert_names}')
    if rec.chunk_id:
        print('chunk_id:', shorten(rec.chunk_id, 120))
    if rec.doc_id:
        print('doc_id  :', shorten(rec.doc_id, 120))
    print('-' * 100)
    if rec.chunk_text:
        print('Chunk:', shorten(rec.chunk_text, MAX_CHARS))
        print('-' * 100)
    print('Q:', shorten(rec.question, MAX_CHARS))
    print('-' * 100)
    print('A:', shorten(rec.answer, MAX_CHARS))
print('=' * 100)


[1/30] quality=0.6588235294117647  experts=['extraction_expert', 'consistency_verification_expert']
chunk_id: 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2024_002001_2024_#ESG_新和成_2024年度环境、社会及公司治理（ESG）报告_2025-04-15_chunk_16
----------------------------------------------------------------------------------------------------
Chunk: 级人才——专家级人才——资深专家级人才——首席专家级人才，以学习路径图的搭建落实人才培养管理序列专业新和成年度培训支出706 万元全体员工总受训时长达人均受训时长达员工培训覆盖率达42.12 小时掌握级精通级专家级资深专家首席专家文化融入、技能训练、目标执行专业训练、训战结合、战术推动攻坚克难、循环赋能、战术构建创新协同、变革孵化、战略推进引领发展新和成员工双晋升通道2024 年，新和成年度培训支出706 万元，全体员工总受训时长达478,910 小时，员工人均受训时长达42.12 小时，员工培训覆盖率达100%新苗育文化，师承启职场新和成为新校招入职员工组织“师承之路”训练营等活动通过线上学习与线下课堂、体能训练、素质拓展和团队协作等形式，新员工们全面深入地学习了企业文化、规章制度、行业知识和职业发展类内容训练营帮助新员工们树立正确的职业态度，掌握了基本的职业技能，使他们更好地认识、认同公司“老师文化”，顺利完成“校园人”向“职场人”的转变新员工开训新和成大学生员工“师承之路”训练营新和成新大学生员工入司训素质拓展活动个人发展计划基层干部扬帆班中层干部启程班高层干部远航班在职干部新晋、后备干部
----------------------------------------------------------------------------------------------------
Q: 验证新和成年度培训支出、员工受训时长、培训覆盖率及其对员工创新能力提升、碳足迹变化或能效改进措施的数据一致性，以及这些指

## Distribution stats: quality score distribution & expert usage frequency

This section performs a **streaming scan** to compute:
- `quality_score` mean, percentiles, and bin counts
- Expert appearance frequency (`selected_experts`)

By default the full file is scanned (may take a few minutes). Use `MAX_LINES` to limit the scan.

In [7]:
# Distribution stats: streaming scan of quality_score / selected_experts
# Full scan may be slow; use MAX_LINES to limit.

from collections import Counter
import numpy as np

MAX_LINES = None  # e.g. 200000 to scan only the first 200k lines

quality_values: list[float] = []
expert_counter = Counter()
n_total = 0
n_with_quality = 0

for i, obj in enumerate(iter_jsonl(QA_FILE), start=1):
    if MAX_LINES is not None and i > MAX_LINES:
        break
    rec = extract_fields(obj)
    n_total += 1
    if rec.quality_score is not None:
        try:
            quality_values.append(float(rec.quality_score))
            n_with_quality += 1
        except Exception:
            pass
    for e in rec.selected_experts:
        expert_counter[str(e)] += 1

print(f"Total rows scanned: {n_total}")
print(f"Rows with quality_score: {n_with_quality}")

# Quality statistics
if quality_values:
    arr = np.array(quality_values, dtype=float)
    print("\nquality_score stats:")
    print(f"  mean: {arr.mean():.4f}")
    for p in (0, 10, 25, 50, 75, 90, 95, 100):
        print(f"  p{p:>3}: {np.percentile(arr, p):.4f}")
else:
    print("\n⚠️ No quality_score values collected (field may be missing or named differently)")

# Quality score bin counts (adjust thresholds to match your quality definition)
bins = [0.0, 0.3, 0.5, 0.7, 1.0]
labels = ['[0.0,0.3)', '[0.3,0.5)', '[0.5,0.7)', '[0.7,1.0]']
counts = [0, 0, 0, 0]

for q in quality_values:
    if q < 0.3:
        counts[0] += 1
    elif q < 0.5:
        counts[1] += 1
    elif q < 0.7:
        counts[2] += 1
    else:
        counts[3] += 1

total = len(quality_values)
print("\nquality_score bins:")
for lab, c in zip(labels, counts):
    pct = (c / total * 100) if total else 0
    print(f"{lab:>10}: {c:>8}  ({pct:6.2f}%)")

扫描行数: 20063
有 quality_score 的行数: 20063

quality_score stats:
  mean: 0.6664
  p  0: 0.0000
  p 10: 0.6061
  p 25: 0.6274
  p 50: 0.6737
  p 75: 0.7208
  p 90: 0.7575
  p 95: 0.7820
  p100: 0.9762

quality_score bins:
 [0.0,0.3):      272  (  1.36%)
 [0.3,0.5):      183  (  0.91%)
 [0.5,0.7):    11612  ( 57.88%)
 [0.7,1.0]:     7996  ( 39.85%)


In [8]:
# Top-N expert usage frequency (uses expert_counter computed in the previous cell)
TOP_N = 30

print(f"Top {TOP_N} experts (by appearances):")
for name, cnt in expert_counter.most_common(TOP_N):
    print(f"  {name:<35} {cnt}")

Top 30 experts (by appearances):
  summary_expert                      18021
  consistency_verification_expert     8566
  classification_expert               1796
  greenwashing_detection_expert       1400
  temporal_analysis_expert            776
  benchmark_comparison_expert         413
  extraction_expert                   156
  standard_alignment_expert           111
  knowledge_graph_expert              90
  analysis_expert                     75
  qa_expert                           15
